## <국립민속박물관-초상 스크래핑>

### 1. 접속 준비 (세션 만들기, CSRF 토큰 만들기)
- 국립민속박물관 검색 기능은 보안을 위해 **CSRF 토큰**이라는 임시 값을 요구
    - 이 요청이 실제로 저 검색 페이지를 열어본 사람이 보낸 게 맞다는 걸 증명하기 위함
- 먼저 검색 페이지에 평범하게 접속(GET)해서, 페이지 안에 숨어있는 CSRF 토큰 값 읽기
- 이때 받은 쿠키(로그인 상태 같은 걸 기억하는 값)와 토큰을 계속 재사용해서 이후 요청 보내기

In [2]:
# 1. 필요한 패키지, 라이브러리 불러오기
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# 공통 함수는 utils.py에 모아두고 여기서 가져다 씀 (01~03 노트북이 다 같은 함수를 공유)
from utils import BASE, get_csrf_token, search_relic_list, get_relic_detail

# 2. URL
SEARCH_LIST_URL = f"{BASE}/user/data/home/101/DataRelicCategoryList.do"
DETAIL_URL = f"{BASE}/user/data/home/101/DataRelicView.do"

# 실제 브라우저처럼 보이도록 User-Agent를 지정해줌 -> 일부 서버는 이게 없으면 차단하기 때문
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

# 3. 쿠키를 자동으로 기억해주는 과정 (쿠키 관리해올 필요 없게)
session = requests.Session()
session.headers.update(HEADERS)

# 4. CSRF 토큰 읽기 (utils.py의 함수 사용, session을 인자로 넘겨줌)
csrf_token = get_csrf_token(session)
print("CSRF 토큰:", csrf_token)

CSRF 토큰: 9ca2d97c-5295-4eb6-9bc3-57f99351183d


### 2. 데이터 수집
- 키워드 기반 수집 -> 첫 페이지 수집 -> 상세 페이지 내용 수집

In [3]:
# 1. 키워드 기반 수집
KEYWORDS = ["초상"]

all_list_items = []
for kw in KEYWORDS:
    all_list_items.extend(search_relic_list(session, csrf_token, kw))

print("검색된 전체 건수(중복 포함):", len(all_list_items))

검색된 전체 건수(중복 포함): 170


In [4]:
# 2. 데이터 프레임 제작
list_df = pd.DataFrame(all_list_items)
list_df = list_df.drop_duplicates(subset="seq").reset_index(drop=True)
print("중복 제거 후 소장품 수:", len(list_df))
list_df.head()  # 여기서 흉배판이 있기 때문에 흉배판은 제함

중복 제거 후 소장품 수: 170


,seq,list_title,searched_keyword
0,PS0100200100108140000000,이선부 초상화첩,초상
1,PS0100200100109555800000,정재기 초상(鄭在箕 肖像),초상
2,PS0100200100104045500000,부부 초상화,초상
3,PS0100200100104045600000,부부 초상화,초상
4,PS0100200100104043800000,의기 계월향 초상(義妓 桂月香 肖像),초상


In [5]:
# 3. 상세 페이지 열기
details = []
for i, row in list_df.iterrows():
    detail = get_relic_detail(session, row["seq"])
    detail["searched_keyword"] = row["searched_keyword"]
    detail["list_title"] = row["list_title"]
    details.append(detail)
    if (i + 1) % 20 == 0:
        print(f"{i + 1} / {len(list_df)} 건 완료")

print("총", len(details), "건")

20 / 170 건 완료
40 / 170 건 완료
60 / 170 건 완료
80 / 170 건 완료
100 / 170 건 완료
120 / 170 건 완료
140 / 170 건 완료
160 / 170 건 완료
총 170 건


### 3. 표로 정리 + 파일 저장

In [6]:
# 1. 데이터프레임 확인
df = pd.DataFrame(details)

# 2. 컬럼 순서 정리
priority_cols = ["searched_keyword", "소장품 명칭", "list_title", "국적/시대", "용도/기능",
                 "크기", "소장품 번호", "내용", "image_url", "detail_url", "seq"]
other_cols = [c for c in df.columns if c not in priority_cols]
ordered_cols = [c for c in priority_cols if c in df.columns] + other_cols
df = df[ordered_cols]

portrait = df[df["소장품 명칭"].str.contains("초상", na=False)]

print(portrait.shape)
portrait.head()

(44, 12)


,searched_keyword,소장품 명칭,list_title,국적/시대,용도/기능,크기,소장품 번호,내용,image_url,detail_url,seq,image_urls
0,초상,이선부 초상화첩,이선부 초상화첩,한국-조선,문화예술-서화-회화-기타,가로 : 37.5 세로 : 50.8,081400,"이선부(李善溥, 1646~1721)의 초상화첩. 1719년(숙종 46) 숙종의 기로...",https://www.nfm.go.kr/common/apiimage/relic/85...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100108140000000,https://www.nfm.go.kr/common/apiimage/relic/85...
1,초상,정재기 초상(鄭在箕 肖像),정재기 초상(鄭在箕 肖像),한국-조선,문화예술-서화-회화-일반회화,세로 : 151.5 가로 : 89.5,095558,"조선 후기의 문신 정재기(鄭在箕, 1811~1879)을 그린 그림. 1869년 5월...",https://www.nfm.go.kr/common/apiimage/relic/85...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100109555800000,https://www.nfm.go.kr/common/apiimage/relic/85...
2,초상,부부 초상화,부부 초상화,한국-광복이후,문화예술-서화-회화-일반회화,가로 : 54 세로 : 134,040455,"부부의 초상화 중 남편을 그린 그림. 화폭(세로 91.7, 가로 34). 정자관에 ...",https://www.nfm.go.kr/common/apiimage/relic/85...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104045500000,https://www.nfm.go.kr/common/apiimage/relic/85...
3,초상,부부 초상화,부부 초상화,한국-광복이후,문화예술-서화-회화-일반회화,가로 : 54 세로 : 134,040456,"부부의 초상화 중 부인을 그린 그림. 화폭(세로 91.7, 가로 34). 흰색 저고...",https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104045600000,https://www.nfm.go.kr/common/apiimage/relic/83...
4,초상,의기 계월향 초상(義妓 桂月香 肖像),의기 계월향 초상(義妓 桂月香 肖像),한국-조선,문화예술-서화-회화-일반회화,세로 : 156.5 가로 : 80.6,040438,평양 기생 계월향(桂月香·?~1592)을 그린 초상화. 그림 상단에는 '의기 계월향...,https://www.nfm.go.kr/common/apiimage/relic/83...,https://www.nfm.go.kr/user/data/home/101/DataR...,PS0100200100104043800000,https://www.nfm.go.kr/common/apiimage/relic/83...


In [7]:
# 3. 파일 저장(엑셀)
portrait.to_excel("../data/nfm_portrait.xlsx", index=False)

# 4. 이미지 저장 (한 유물에 사진이 여러 장이면 -1, -2 ... 붙여서 다 저장함)
portrait = portrait.reset_index(drop=True) 

for i, row in portrait.iterrows():
    seq = row["seq"]
    image_urls = str(row["image_urls"]).split("; ") if pd.notna(row["image_urls"]) else []

    for idx, image_url in enumerate(image_urls):
        suffix = "" if len(image_urls) == 1 else f"-{idx + 1}"  # 여러 장일 때만 -1, -2 ... 붙임
        filepath = f"../image/portrait/{seq}{suffix}.jpg"

        resp = session.get(image_url, timeout=30)
        with open(filepath, "wb") as f:
            f.write(resp.content)

    if (i + 1) % 44 == 0:
        print(f"{i + 1} / {len(portrait)} 완료")

    time.sleep(0.5)

44 / 44 완료
